In [ ]:
import logging
class AILogger:
    

    def __init__(self, log_dir: str = "logs"):
        
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(parents=True, exist_ok=True)

        # Filename: logs/ai_log_2026-06-15_10-30-00.jsonl
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        self.log_file = self.log_dir / f"ai_log_{timestamp}.jsonl"

        # Standard Python logging for console output
        logging.basicConfig(
            level=logging.INFO,
            format="%(asctime)s | %(levelname)-8s | %(message)s",
            datefmt="%H:%M:%S",
        )
        self.console = logging.getLogger("AI-Script")

    def log_call(
        self,
        model: str,
        prompt: str,
        response_text: str,
        prompt_tokens: int = 0,
        completion_tokens: int = 0,
        total_tokens: int = 0,
        duration_ms: float = 0.0,
        chain_step: Optional[str] = None,
        batch_id: Optional[str] = None,
        status: str = "success",
        extra: Optional[dict] = None,
    ):
        """
        Log ONE AI interaction.

        This is called after EVERY AI API call, whether success or error.
        It writes a structured JSON object to the log file and prints
        a one-line summary to the console.

        Args:
            model: Model name (e.g., 'gpt-4o-mini')
            prompt: The input text sent to the AI
            response_text: The AI's response
            prompt_tokens: Number of input tokens
            completion_tokens: Number of output tokens
            total_tokens: Total tokens
            duration_ms: Time taken in milliseconds
            chain_step: Which step in a chain (e.g., 'summarize', 'translate')
            batch_id: Batch identifier for grouping related calls
            status: 'success' or 'error'
            extra: Any additional data to include
        """
        # ── Build structured log entry ─────────────────────
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "model": model,
            "chain_step": chain_step,
            "batch_id": batch_id,
            "prompt_preview": prompt[:200] + ("..." if len(prompt) > 200 else ""),
            "response_preview": response_text[:200] + ("..." if len(response_text) > 200 else ""),
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens,
            "duration_ms": round(duration_ms, 2),
            "status": status,
        }
        if extra:
            log_entry.update(extra)

        # ── Write to JSONL file (append mode) ─────────────
        with open(self.log_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(log_entry) + "\n")

        # ── Console output (one line per call) ────────────
        tokens_str = f" [{prompt_tokens}in + {completion_tokens}out = {total_tokens}tok]"
        chain_str = f" [{chain_step}]" if chain_step else ""
        batch_str = f" [batch:{batch_id}]" if batch_id else ""
        duration_str = f" [{duration_ms:.0f}ms]"

        status_icon = "✓" if status == "success" else "✗"
        self.console.info(f"{status_icon} {chain_str}{batch_str}{tokens_str}{duration_str}")

    def summary(self) -> dict:
        """
        Read the current log file and produce a summary.

        Returns:
            dict with: total_calls, errors, total_tokens, avg_duration_ms, log_file
        """
        if not self.log_file.exists():
            return {"error": "No log file found"}

        calls = 0
        errors = 0
        total_tok = 0
        durations = []

        with open(self.log_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                entry = json.loads(line)
                calls += 1
                if entry.get("status") == "error":
                    errors += 1
                total_tok += entry.get("total_tokens", 0)
                durations.append(entry.get("duration_ms", 0))

        avg_duration = sum(durations) / len(durations) if durations else 0
        return {
            "log_file": str(self.log_file),
            "total_calls": calls,
            "errors": errors,
            "total_tokens": total_tok,
            "avg_duration_ms": round(avg_duration, 2),
        }

    def export_csv(self, csv_path: str = "logs/ai_log_export.csv"):
        """
        Export the JSONL log to CSV for spreadsheet analysis.

        This lets you open the log in Excel/Google Sheets
        for charting and analysis.
        """
        if not self.log_file.exists():
            print("No log file to export.")
            return

        with open(self.log_file, "r", encoding="utf-8") as f:
            entries = [json.loads(line) for line in f if line.strip()]

        if not entries:
            print("No log entries to export.")
            return

        fieldnames = list(entries[0].keys())
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(entries)
        print(f"📊 Exported {len(entries)} log entries to {csv_path}")


# Create a global logger instance — all functions share this.
ai_logger = AILogger()

print("✅ AILogger created — logs will be saved to: logs/ai_log_*.jsonl")